In [29]:
import os
import json
import rasterio
import pandas as pd
from tqdm import tqdm

def validate_output_files(base_folder, debug=False):
    """
    Validates that all expected output files are present and valid within each subdirectory of the base folder.

    Parameters:
    - base_folder (str): The path to the base directory containing subdirectories with output files.
    - debug (bool): If True, prints expected vs actual filenames for debugging.

    Returns:
    - None: Prints a validation summary directly.
    """

    expected_suffixes = [
        "ancillary",
        "ancillary.hdr",
        "brdf_coeffs__envi.json",
        "config__anc.json",
        "config__envi.json",
        "__envi",
        "__envi.hdr",
        "__envi_mask",
        "__envi_mask.hdr",
        "__envi_mask_spectral_data.csv",
        "__envi_masked",
        "__envi_masked.aux.xml",
        "__envi_masked.hdr",
        "__envi_masked_spectral_data.csv",
        "topo_coeffs__envi.json",
        "masked",
        "masked.aux.xml",
        "masked.hdr",
        "masked_spectral_data.csv",
    ]

    subdirectories = [
        os.path.join(base_folder, d) for d in os.listdir(base_folder)
        if os.path.isdir(os.path.join(base_folder, d)) and not d.startswith('.ipynb_checkpoints')
    ]

    if not subdirectories:
        print(f"❌ No subdirectories found in the base folder: {base_folder}")
        return

    print(f"🔍 Starting validation of output files in base folder: {base_folder}\n")

    for subdir in tqdm(subdirectories, desc="Validating subdirectories"):
        subdir_name = os.path.basename(subdir)

        # Ensure we don't duplicate "_directional_reflectance"
        if "_directional_reflectance" in subdir_name:
            base_name = subdir_name
        else:
            base_name = f"{subdir_name}_directional_reflectance"

        # Build expected filenames:
        expected_files = set()
        for suffix in expected_suffixes:
            if suffix.startswith("_"):
                filename = f"{base_name}{suffix}"
            else:
                filename = f"{base_name}_{suffix}"
            expected_files.add(os.path.join(subdir, filename))

        # Get actual files in the directory, ignoring hidden/system files
        existing_files = {os.path.join(subdir, f.strip()) for f in os.listdir(subdir) if not f.startswith('.')}

        # Debug: Print expected vs actual filenames
        if debug:
            print(f"\n📂 Subdirectory: {subdir_name}")
            print("Expected Files:")
            for file in sorted(expected_files):
                print(f"  - {os.path.basename(file)}")
            print("\nActual Files:")
            for file in sorted(existing_files):
                print(f"  - {os.path.basename(file)}")

        # Find missing files
        missing_files = sorted(expected_files - existing_files)
        invalid_files = []

        # Validate file integrity
        for file_path in existing_files:
            try:
                if file_path.endswith('.hdr'):
                    # HDR files: existence is enough
                    pass
                elif file_path.endswith(('.img', '_envi', '_mask')):
                    # Validate raster files
                    with rasterio.open(file_path) as src:
                        _ = src.meta  # Ensure metadata is accessible
                elif file_path.endswith('.csv'):
                    pd.read_csv(file_path, nrows=5)  # Validate CSV files by reading a few rows
                elif file_path.endswith('.json'):
                    with open(file_path, 'r') as f:
                        json.load(f)  # Validate JSON files by loading them
            except Exception as e:
                invalid_files.append(f"{os.path.basename(file_path)} (Error: {str(e)})")

        # Print summary for each subdirectory
        if not missing_files and not invalid_files:
            print(f"✅ Subdirectory: {subdir_name} - All expected files are present and valid.\n")
        else:
            print(f"❌ Subdirectory: {subdir_name}")
            if missing_files:
                print("   🚨 Missing Files:")
                for missing in missing_files:
                    print(f"     - {os.path.basename(missing)}")
            if invalid_files:
                print("   ⚠️ Invalid Files:")
                for invalid in invalid_files:
                    print(f"     - {invalid}")
            print()  # Blank line for readability

# Example usage:
# validate_output_files('/path/to/base_folder', debug=True)



In [44]:
validate_output_files('YELL_2023_07', debug=False)


🔍 Starting validation of output files in base folder: YELL_2023_07



Validating subdirectories:   8%|▊         | 1/13 [00:00<00:02,  5.14it/s]

✅ Subdirectory: NEON_D12_YELL_DP1_L065-1_20230712_directional_reflectance - All expected files are present and valid.



Validating subdirectories:  15%|█▌        | 2/13 [00:00<00:02,  5.10it/s]

✅ Subdirectory: NEON_D12_YELL_DP1_L074-1_20230718_directional_reflectance - All expected files are present and valid.



...

✅ Subdirectory: NEON_D12_YELL_DP1_L033-1_20230715_directional_reflectance - All expected files are present and valid.

✅ Subdirectory: NEON_D12_YELL_DP1_L040-1_20230715_directional_reflectance - All expected files are present and valid.



Validating subdirectories:  85%|████████▍ | 11/13 [00:02<00:00,  4.76it/s]

✅ Subdirectory: NEON_D12_YELL_DP1_L050-1_20230712_directional_reflectance - All expected files are present and valid.



Validating subdirectories: 100%|██████████| 13/13 [00:02<00:00,  4.79it/s]

✅ Subdirectory: NEON_D12_YELL_DP1_L067-1_20230712_directional_reflectance - All expected files are present and valid.

✅ Subdirectory: NEON_D12_YELL_DP1_L075-1_20230718_directional_reflectance - All expected files are present and valid.

